# RenderForge TTS — Audio Generator

Generate narration audio for RenderForge videos using **Qwen3-TTS**.

### Workflow
1. Upload `scripts.json` (generated by `npx tsx content/generate-scripts.ts`)
2. Choose voice style & speaker
3. Generate `.wav` files for each section
4. Download zip → place in `content/audio/` → run `audio-sync.ts`

### IMPORTANT: Enable GPU Runtime
**Before running anything:**
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** (free tier) or **A100** (Colab Pro)
3. Click **Save**

### Available Models

| Model | Size | GPU | Modes |
|-------|------|-----|-------|
| `0.6B-CustomVoice` | ~1.2GB | T4 (free) | 9 built-in speakers + style instructions |
| `0.6B-Base` | ~1.2GB | T4 (free) | Voice cloning from reference audio |
| `1.7B-VoiceDesign` | ~3.4GB | A100 | Free-form voice description (most flexible) |
| `1.7B-CustomVoice` | ~3.4GB | A100 | 9 speakers + style instructions |
| `1.7B-Base` | ~3.4GB | A100 | Voice cloning |

> **Free tier (T4)**: Use `CustomVoice 0.6B` with speaker **Aiden** or **Ryan** for English.
> **VoiceDesign** (describe any voice) requires 1.7B → A100 only.

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Check GPU is available
import subprocess
gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu_check.returncode == 0:
    gpu_info = gpu_check.stdout.strip()
    print(f"GPU detected: {gpu_info}")
    if 'A100' in gpu_info:
        print("A100 — all models available (0.6B and 1.7B)")
    elif 'T4' in gpu_info:
        print("T4 — use 0.6B models (CustomVoice or Base)")
    else:
        print("Other GPU — try 0.6B first, upgrade if OOM")
else:
    print("NO GPU DETECTED!")
    print("Go to Runtime → Change runtime type → Select T4 GPU → Save")
    print("Then restart and run this cell again.")
    raise RuntimeError("GPU required. Enable GPU runtime first.")

In [ ]:
# Install dependencies
!pip install -U qwen-tts soundfile numpy -q

# Install sox for audio processing (optional, suppresses warning)
!apt-get install -y sox libsox-dev > /dev/null 2>&1 && echo "sox installed" || echo "sox not available (non-critical)"

# Optional: Flash Attention for faster generation
!pip install -U flash-attn --no-build-isolation -q 2>/dev/null || echo "Flash Attention not available — will use default attention (still works fine)"

print("\nDependencies ready!")

## Step 2: Upload scripts.json

Upload the `scripts.json` file generated locally by:
```bash
npx tsx content/generate-scripts.ts --limit 10
```

In [ ]:
from google.colab import files
import json

# Upload scripts.json
print("Upload your scripts.json file:")
uploaded = files.upload()

scripts = json.loads(list(uploaded.values())[0].decode('utf-8'))
total_sections = sum(len(s['sections']) for s in scripts)

print(f"\nLoaded {len(scripts)} posts")
print(f"Total audio sections to generate: {total_sections}")
print(f"\nPosts:")
for s in scripts[:5]:
    print(f"  [{s['postId']}] {s['template']} — {s['title'][:50]}")
if len(scripts) > 5:
    print(f"  ... and {len(scripts) - 5} more")

## Step 3: Configure Voice

### Available English Speakers (CustomVoice)
| Speaker | Description |
|---------|-------------|
| **Aiden** | Sunny American male — great for motivational content |
| **Ryan** | Dynamic male with rhythm — great for energetic content |

### Voice Modes
| Mode | Model Needed | Description |
|------|-------------|-------------|
| `custom` | 0.6B-CustomVoice (T4 OK) | Pick a speaker + style instruction |
| `clone` | 0.6B-Base (T4 OK) | Clone from reference audio |
| `design` | 1.7B-VoiceDesign (A100 only) | Describe any voice freely |

In [ ]:
# ═══════════════════════════════════════════
#  VOICE CONFIGURATION
# ═══════════════════════════════════════════

# Choose mode: "custom" (recommended for T4), "clone", or "design" (A100 only)
VOICE_MODE = "custom"

# ── CustomVoice Settings (works on free T4!) ──
# Speaker: "Aiden" (sunny American male) or "Ryan" (dynamic rhythmic male)
SPEAKER = "Aiden"

# Style instruction — controls emotion and delivery
VOICE_INSTRUCT = (
    "Speak with confidence and energy, like a motivational speaker. "
    "Clear pronunciation, engaging tone, medium-fast pace."
)

# ── Voice Design Settings (A100 only) ──
DESIGN_INSTRUCT = (
    "A confident, energetic young male voice with a motivational tone. "
    "Clear pronunciation, medium-fast pace, slightly deep pitch. "
    "Sounds like a successful entrepreneur giving advice."
)

# ── Voice Clone Settings ──
CLONE_REF_AUDIO = None  # Set in Step 3b
CLONE_REF_TEXT = ""     # Set in Step 3b

# ── Model Size ──
# "0.6B" for free T4 | "1.7B" for A100
MODEL_SIZE = "0.6B"
LANGUAGE = "English"

print(f"Voice mode:  {VOICE_MODE}")
print(f"Model size:  {MODEL_SIZE}")
print(f"Speaker:     {SPEAKER}")
print(f"Language:    {LANGUAGE}")
print(f"Style:       {VOICE_INSTRUCT[:60]}...")

## Step 3b: (Optional) Upload Reference Audio for Voice Clone

Only run this cell if `VOICE_MODE = "clone"`. Upload a `.wav` file with 5-15 seconds of clear speech.

In [ ]:
# Only run this if VOICE_MODE = "clone"
if VOICE_MODE == "clone":
    print("Upload a reference audio file (.wav, 5-15 seconds of clear speech):")
    ref_upload = files.upload()
    ref_filename = list(ref_upload.keys())[0]
    CLONE_REF_AUDIO = ref_filename
    print(f"\nReference audio: {ref_filename}")
    CLONE_REF_TEXT = input("Enter transcription of the reference audio: ")
    print(f"Reference text: {CLONE_REF_TEXT}")
else:
    print("Skipped — not using clone mode")

## Step 4: Load Model

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

# Select model based on mode and size
MODEL_MAP = {
    "custom": f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice",
    "clone":  f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base",
    "design": f"Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",  # design is 1.7B only
}
model_name = MODEL_MAP[VOICE_MODE]

print(f"Loading model: {model_name}")
print("This may take 2-5 minutes on first run (downloading weights)...\n")

# Try flash attention, fall back to default
try:
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    print("Loaded with Flash Attention 2")
except Exception as e:
    print(f"Flash Attention not available ({str(e)[:50]}), using default...")
    model = Qwen3TTSModel.from_pretrained(
        model_name,
        device_map="cuda:0",
        dtype=torch.bfloat16,
    )
    print("Loaded with default attention")

print(f"\nModel: {model_name.split('/')[-1]}")
print(f"Device: {next(model.parameters()).device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 5: Generate Audio

Generates a `.wav` file for each section + a `full.wav` combining all sections per post.

In [ ]:
import soundfile as sf
import numpy as np
import os
import time

os.makedirs("audio", exist_ok=True)

def generate_audio(text):
    """Generate audio using configured voice mode."""
    if VOICE_MODE == "custom":
        wavs, sr = model.generate_custom_voice(
            text=text,
            language=LANGUAGE,
            speaker=SPEAKER,
            instruct=VOICE_INSTRUCT,
        )
    elif VOICE_MODE == "clone":
        wavs, sr = model.generate_voice_clone(
            text=text,
            language=LANGUAGE,
            ref_audio=CLONE_REF_AUDIO,
            ref_text=CLONE_REF_TEXT,
        )
    elif VOICE_MODE == "design":
        wavs, sr = model.generate_voice_design(
            text=text,
            language=LANGUAGE,
            instruct=DESIGN_INSTRUCT,
        )
    return wavs[0], sr

# Count total sections
total = sum(len(s['sections']) for s in scripts)
done = 0
failed = []
start_time = time.time()

print(f"Generating {total} audio files for {len(scripts)} posts...")
print(f"Mode: {VOICE_MODE} | Speaker: {SPEAKER} | Model: {MODEL_SIZE}")
print("=" * 60)

for post in scripts:
    post_id = post['postId']
    post_dir = os.path.join("audio", post_id)
    os.makedirs(post_dir, exist_ok=True)

    print(f"\n[{post_id}] {post['template']} — {post['title'][:50]}")

    post_wavs = []
    sr = None

    for section in post['sections']:
        done += 1
        elapsed = time.time() - start_time
        per_item = elapsed / done if done > 0 else 0
        eta = per_item * (total - done)
        eta_str = f"{int(eta // 60)}m {int(eta % 60)}s" if eta > 60 else f"{int(eta)}s"

        output_path = os.path.join("audio", section['audioFile'])
        text = section['text']

        print(f"  [{done}/{total}] {section['key']}: \"{text[:55]}{'...' if len(text) > 55 else ''}\" (ETA: {eta_str})", end="")

        try:
            wav, current_sr = generate_audio(text)
            sf.write(output_path, wav, current_sr)
            duration = len(wav) / current_sr
            print(f" -> {duration:.1f}s")

            post_wavs.append(wav)
            if sr is None:
                sr = current_sr
        except Exception as e:
            print(f" FAILED: {str(e)[:80]}")
            failed.append({'post': post_id, 'section': section['key'], 'error': str(e)})

    # Save full narration (all sections concatenated)
    if post_wavs and sr:
        # Add 0.3s silence between sections for natural pacing
        silence = np.zeros(int(sr * 0.3), dtype=post_wavs[0].dtype)
        parts = []
        for i, wav in enumerate(post_wavs):
            parts.append(wav)
            if i < len(post_wavs) - 1:
                parts.append(silence)
        full_wav = np.concatenate(parts)
        full_path = os.path.join(post_dir, "full.wav")
        sf.write(full_path, full_wav, sr)
        full_duration = len(full_wav) / sr
        print(f"  >> Full narration: {full_duration:.1f}s")

total_time = time.time() - start_time
print("\n" + "=" * 60)
print(f"DONE! {done - len(failed)}/{total} audio files in {int(total_time // 60)}m {int(total_time % 60)}s")
if failed:
    print(f"\nFailed ({len(failed)}):")
    for f in failed:
        print(f"  {f['post']}/{f['section']}: {f['error'][:80]}")

## Step 6: Preview Audio

Listen to samples before downloading.

In [ ]:
from IPython.display import Audio, display
import os

# Preview first post's full narration
first_post = scripts[0]['postId']
preview_path = f"audio/{first_post}/full.wav"

if os.path.exists(preview_path):
    print(f"Preview: {first_post} — {scripts[0]['title']}")
    print(f"Full narration:")
    display(Audio(preview_path))
    
    # Also play individual sections
    print(f"\nIndividual sections:")
    for sec in scripts[0]['sections']:
        sec_path = f"audio/{sec['audioFile']}"
        if os.path.exists(sec_path):
            print(f"  {sec['key']}: \"{sec['text'][:50]}...\"")
            display(Audio(sec_path))
else:
    print("No audio files found. Run Step 5 first.")

## Step 7: Download

Downloads a zip file. Then on your local machine:
```bash
# Extract into renderforge/content/ so structure is:
# content/audio/day01-post1/intro.wav
# content/audio/day01-post1/headline.wav
# etc.

# Then run the sync pipeline:
npx tsx content/audio-sync.ts --limit 10

# Output: output/synced/story/*.mp4
```

In [ ]:
import shutil
import os

# Show what was generated
total_size = 0
file_count = 0
for root, dirs, fnames in os.walk("audio"):
    for f in fnames:
        if f.endswith('.wav'):
            total_size += os.path.getsize(os.path.join(root, f))
            file_count += 1

print(f"Audio files: {file_count}")
print(f"Total size:  {total_size / 1024 / 1024:.1f} MB")

# Create zip
print("\nCreating zip...")
shutil.make_archive("renderforge-audio", "zip", ".", "audio")
zip_size = os.path.getsize("renderforge-audio.zip") / 1024 / 1024
print(f"Zip: renderforge-audio.zip ({zip_size:.1f} MB)")

# Download
print("\nStarting download...")
from google.colab import files
files.download("renderforge-audio.zip")

print("\n" + "=" * 60)
print("NEXT STEPS:")
print("1. Extract zip into renderforge/content/ directory")
print("   (files should be at content/audio/day01-post1/intro.wav)")
print("2. Run: npx tsx content/audio-sync.ts --limit 10")
print("3. Find synced videos in output/synced/story/")
print("=" * 60)

---

## Voice Reference

### Built-in Speakers (CustomVoice)
| Speaker | Description | Language |
|---------|-------------|----------|
| **Aiden** | Sunny American male | English |
| **Ryan** | Dynamic male with rhythm | English |
| Vivian | Bright young female | Chinese |
| Serena | Warm, gentle young female | Chinese |
| Uncle_Fu | Seasoned male, mellow | Chinese |
| Dylan | Youthful Beijing male | Chinese |
| Eric | Lively Chengdu male | Chinese |
| Ono_Anna | Playful Japanese female | Japanese |
| Sohee | Warm Korean female | Korean |

### Style Instructions (VOICE_INSTRUCT examples)
| Channel | Instruction |
|---------|-------------|
| **Your Last Dollar** | `"Speak with confidence and energy, like a motivational speaker. Clear, engaging, medium-fast pace."` |
| **KnockKnockZoo** | `"Speak in a cheerful, playful tone like telling jokes to kids. Bright, fun, slightly exaggerated."` |
| **Stoicism** | `"Speak slowly and deliberately, with calm authority. Like a wise mentor sharing deep wisdom."` |
| **Horror** | `"Speak quietly with a slightly eerie tone. Slow pace with dramatic pauses. Unsettling."` |
| **Tech News** | `"Speak clearly and professionally. Informative, medium pace, like a tech journalist."` |
| **Dubai Luxury** | `"Speak smoothly and elegantly. Sophisticated tone, medium-slow pace, refined."` |